# Canonical Model 03 · PRT and Parallel Splitting

Build and run the 50×50, four-layer canonical DISV valley model, trace particles
with MF6 **PRT**, and split the same simulation across eight real MPI processes.

> **Validation goal:** every stage must terminate normally, preserve the valley
> lake as a contiguous LAK feature, and produce reconstructable results.

In [ ]:
import sys
from pathlib import Path

# Make the in-repo `src/` importable when myflopy is not pip-installed.
src = Path.cwd().parents[2] / 'src'
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))
import pandas as pd
import myflopy as mf
from myflopy.modflow.mf6.canonical_example import representative_cells
from canonical_notebook_style import notebook_header

notebook_header('03', 'PRT and Parallel', 'Trace groundwater movement and run the same model across MPI partitions.')

root = Path('../artifacts/canonical_prt_parallel')
config = mf.CanonicalModelConfig.validation()
model = mf.build_canonical_model(root / 'gwf', config=config)
success, gwf_report = model.run_simulation()
assert success, '\n'.join(gwf_report[-30:])

pd.Series({
    'rows': config.nrow, 'columns': config.ncol, 'layers': config.nlay,
    'total_3d_cells': config.nrow * config.ncol * config.nlay,
    'gwf_termination': gwf_report[-1],
}, name='canonical GWF')

## MF6 PRT

PRT consumes the completed GWF head, budget, and binary-grid files. Particles are
released along the up-valley mountain front and tracked forward through the
regional flow field. Tracking stops at the end of available flow output by
default, preventing an unbounded final-time-step run.

In [ ]:
releases = mf.PRTReleasePoints.from_cells(model, representative_cells(config)['releases'])
prt = model.particle_tracking.prt(
    workspace=root / 'prt',
    release_points=releases,
    porosity=0.25,
    extend_tracking=False,
)
prt_results = prt.run(silent=True)
assert prt_results.success

# `pathlines` is a view, not a frame: .get() returns the normalized records --
# every raw track-CSV column plus cell/layer/travel_time/release_group/particle.
tracks = prt_results.pathlines.get()
assert not tracks.empty

pd.Series({
    'release_points': len(releases.packagedata),
    'pathline_records': len(tracks),
    'tracked_particles': tracks['particle'].nunique(),
}, name='PRT results')

**What to look for:** particles released at the mountain front migrate down the
valley axis, drawn along the high-K paleochannel toward the stream corridor and
the terminal lake — the advective picture behind the head maps in 02.

In [ ]:
# One library figure instead of a hand-rolled redraw: a polyline per particle
# over the final-period water table, with per-vertex hover (cell, elapsed time,
# layer) and the house pan/scroll-zoom styling.
prt_results.pathlines.map(
    per=config.nper - 1,
    layer=0,
    title='PRT pathlines over the water table',
)

## Lake-safe partitions

Partition cuts route around the valley lake. This avoids creating zero-area local
LAK fragments while keeping every model partition contiguous. We prepare and
validate 2 through 8 partitions without writing them.

In [ ]:
partition_rows = []
for nparts in range(2, 9):
    candidate_mask = mf.canonical_partition_mask(model, nparts)
    prepared = model.parallel.split_model(workspace=root / f'split_{nparts}', mask=candidate_mask, write=False)
    validation = prepared.validate()
    partition_rows.append({
        'partitions': nparts,
        'all_contiguous': bool(validation['contiguous'].all()),
        'minimum_columns': int(validation['columns'].min()),
        'maximum_columns': int(validation['columns'].max()),
    })
    print(f'Prepared and validated {nparts} contiguous partitions.')

partition_summary = pd.DataFrame(partition_rows).set_index('partitions')
assert partition_summary['all_contiguous'].all()
partition_summary

## Eight-process MPI run

The final partition set is written and run with eight MPI workers. The current
Python environment's matching `mf6` and `mpiexec` executables are discovered
automatically.

In [ ]:
mask = mf.canonical_partition_mask(model, 8)
split = model.parallel.split_model(workspace=root / 'split_8', mask=mask)
split.summary()

In [ ]:
# RUN_MPI gates the 8-process MPI launch + reconstruction below. Saved
# outputs are from a completed run. Set RUN_MPI = True to regenerate them
# (needs a matching mpiexec on PATH; ~1-2 min).
RUN_MPI = False
if RUN_MPI:
    parallel_success, parallel_report = split.run(processors=split.nparts, write=False, silent=True)
    assert parallel_success, '\n'.join(parallel_report[-30:])
    assert any('PARALLEL mode' in line for line in parallel_report)
    display(pd.Series({
        'processors': split.nparts,
        'parallel_mode_confirmed': True,
        'termination': parallel_report[-1],
    }, name='MPI run'))


## Reconstructed results

Split outputs are reconstructed onto the original DISV grid and compared with the
source-model heads. Small numerical differences are expected; large differences
indicate a split or exchange problem.

In [ ]:
if RUN_MPI:
    head_comparison = split.compare_heads()
    assert head_comparison.loc[0, 'max_absolute_error'] < 0.1, head_comparison
    display(head_comparison)


## Result

The same 50×50 canonical valley model now completes as a source GWF simulation, a
finite MF6 PRT simulation, and an eight-process MPI split simulation — with the
valley lake preserved and heads reconstructed to within tolerance. Every
code-cell output above is part of the validation record.

Continue to **04 · PEST and Results** to calibrate this same model.